In [1]:
import numpy as np
import pandas as pd

from astroML.datasets import fetch_sdss_specgals
from sklearn.model_selection import train_test_split
from astroML.utils.decorators import pickle_results

In [2]:
data = data = fetch_sdss_specgals()
df = pd.DataFrame(data)

pd.set_option('display.float_format', lambda x: '%.9f' % x)

In [3]:
# Calculate the color indices
df['u-g'] = df['modelMag_u'] - df['modelMag_g']
df['g-r'] = df['modelMag_g'] - df['modelMag_r']
df['r-i'] = df['modelMag_r'] - df['modelMag_i']
df['i-z'] = df['modelMag_i'] - df['modelMag_z']

In [4]:
# Dataset versions
configs = [
    {'version': 'v1', 'mag_error': 1, 'z_min': 0.25, 'clean': True},
    {'version': 'v2', 'mag_error': 2.5, 'z_min': 0.2, 'clean': True},
    {'version': 'v3', 'mag_error': 5, 'z_min': 0.1, 'clean': True},
    {'version': 'v4', 'mag_error': None, 'z_min': None, 'clean': False} # No data cleaning
]

# Loop through each version
for config in configs:
    v = config['version']
    
    # Clean the data (conditionally)
    if config['clean']:
        mag_err = config['mag_error']
        z_min = config['z_min']
        
        df_clean = df[
            (df['modelMagErr_u'] < mag_err) &
            (df['modelMagErr_g'] < mag_err) &
            (df['modelMagErr_r'] < mag_err) &
            (df['modelMagErr_i'] < mag_err) &
            (df['modelMagErr_z'] < mag_err) &
            (df['z'] > z_min)
        ]
    else:
        df_clean = df.copy()
        
    print(f"Shape: {df_clean.shape}")
    
    # train/val/test split
    X = df_clean[['u-g', 'g-r', 'r-i', 'i-z']]
    y = df_clean['z']
    
    # Define the file path based on the current version
    file_path = f'data/dataset_split_{v}.pkl'
    
    @pickle_results(file_path)
    def split_data(X_data, y_data):
        X_train, X_temp, y_train, y_temp = train_test_split(X_data, y_data, test_size=0.3, random_state=42)
        X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)
        return X_train, X_val, X_test, y_train, y_val, y_test

    # Run the function to generate and cache the splits
    X_train, X_val, X_test, y_train, y_val, y_test = split_data(X, y)

Shape: (10530, 47)
@pickle_results: using precomputed results from 'data/dataset_split_v1.pkl'
Shape: (43621, 47)
@pickle_results: using precomputed results from 'data/dataset_split_v2.pkl'
Shape: (348113, 47)
@pickle_results: using precomputed results from 'data/dataset_split_v3.pkl'
Shape: (661598, 47)
@pickle_results: using precomputed results from 'data/dataset_split_v4.pkl'


### Dataset v1:
- mag_error = 1
- z > 0.25

### Dataset v2:
- mag_error = 2.5
- z > 0.2

### Dataset v3:
- mag_error = 5
- z > 0.1

### Dataset v4: 
df_clean = df (no data cleaning)